# Notebook 03 — Análisis semántico

**Objetivo:** usar modelos de inteligencia artificial para analizar el **significado** del texto,
no solo su estructura. Mientras el notebook 02 respondía preguntas de "quién se conecta con quién",
este notebook responde "qué dicen" y "cómo escriben".

**Técnicas en este notebook (de mayor a menor complejidad):**

| Técnica | ¿Qué hace? | Modelo |  
|---------|-----------|--------|  
| **Embeddings** | Convierte texto en vectores numéricos que capturan significado | qwen3-embedding (4096D) |  
| **UMAP + HDBSCAN** | Reduce dimensiones y detecta clusters de usuarios similares | Matemáticas |  
| **BERTopic** | Descubre automáticamente los temas principales del foro | Estadística + embeddings |  
| **NER** | Extrae entidades nombradas (personas, organizaciones, ideologías) | qwen2.5:14b |  
| **Perfilado** | Sintetiza el perfil de cada actor clave | qwen2.5:14b |  
| **Burrows' Delta** | Mide similitud de estilo de escritura entre usuarios | Estadística clásica |  

**Prerequisito:** `01_ingenieria_datos.ipynb` debe haberse ejecutado para tener los parquets.
Los modelos de embeddings y NER requieren **Ollama** corriendo localmente.

**Nota sobre los outputs precomputados:** las celdas de embeddings y NER pueden tardar
horas en ejecutarse. Si ya existen archivos de caché en `results/ironmarch/`, se cargan
automáticamente. Los outputs ya calculados se muestran en el notebook original `_archive/03_embeddings_profiling.ipynb`.

## 1. Imports y carga de datos

In [ ]:
import sys
import json
import re
from pathlib import Path
from datetime import datetime
from collections import Counter
from itertools import combinations

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import spearmanr
from tqdm.auto import tqdm
import umap as umap_lib
import hdbscan
import ollama
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

from src.utils import RESULTS_DIR
from src.embeddings import embed_users, compute_actor_centroids

plt.style.use('dark_background')
IM_RESULTS = RESULTS_DIR / 'ironmarch'

# Cargar parquets limpios
posts_clean = pd.read_parquet(IM_RESULTS / 'posts_clean.parquet')
users_clean = pd.read_parquet(IM_RESULTS / 'users_clean.parquet')

uid_to_name = dict(zip(users_clean['userid'], users_clean['username_raw']))

print(f'Posts: {len(posts_clean):,}  |  Usuarios: {len(users_clean):,}')
print(f'Cols posts: {list(posts_clean.columns)}')

## 2. Embeddings: representar usuarios como vectores

### ¿Qué es un embedding?

Un **embedding** es la traducción de texto a un vector numérico (una lista de números)
que captura el significado semántico. Dos textos con significado similar tendrán vectores
cercanos en el espacio vectorial.

Por ejemplo:
- "Hitler fue un líder" y "el Führer fue decisivo" → vectores muy cercanos
- "Hitler fue un líder" y "me gusta la pizza" → vectores muy lejanos

Usamos `qwen3-embedding` que produce vectores de **4096 dimensiones**.
Para cada usuario, generamos un vector que representa el estilo y contenido de todos sus posts.

### El experimento metodológico: ¿cuántos posts necesitamos por usuario?

Probamos 5 estrategias distintas y medimos cuán similares son los resultados:

| Estrategia | Método | Posts procesados | Tiempo real | Correlación vs full (ρ) |
|---|---|---|---|---|
| A | Concatenar todos los posts → 1 vector | 157,779 | ~12 min | — |
| B | 1 embedding por post, promedio (completo) | 157,779 | ~9 h | referencia |
| C | Top-50 posts más largos por usuario | ≤41,800 | 2h 35min | **ρ=0.799** |
| D | Top-100 posts más largos por usuario | 41,319 | 3h 33min | **ρ=0.860** |
| E | Top-150 posts más largos por usuario | 53,193 | 4h 12min | **ρ=0.896** |

**Conclusión:** el salto de C→D (+37min, +6 puntos ρ) es más eficiente que D→E (+39min, +4 puntos ρ).
Para análisis exploratorio, **D (top-100) es el mejor balance** entre fidelidad y costo computacional.

In [ ]:
# Función para cargar el archivo .npz más reciente que coincida con un patrón
# Los archivos .npz son arrays numpy comprimidos — ideal para guardar matrices grandes
def load_latest(pattern: str):
    """Carga el archivo .npz más reciente que coincida con el patrón.
    Retorna un diccionario con los arrays, o None si no existe ninguno."""
    matches = sorted(IM_RESULTS.glob(pattern))
    if not matches:
        return None
    path = matches[-1]
    print(f'Cargando caché: {path.name}')
    return dict(np.load(path, allow_pickle=True))

def save_timestamped(data: dict, section: str, name: str):
    """Guarda un diccionario de arrays con timestamp en el nombre del archivo."""
    ts  = datetime.now().strftime('%Y%m%d_%H%M%S')
    out = IM_RESULTS / f'{section}_{name}_{ts}.npz'
    np.savez_compressed(out, **data)
    print(f'Guardado: {out.name}')
    return out

# Preparar posts para embedding
text_col = 'pagetext' if 'pagetext' in posts_clean.columns else 'message'
posts_embed = posts_clean[['userid', text_col]].copy()
posts_embed = posts_embed.dropna(subset=[text_col])
posts_embed = posts_embed[posts_embed[text_col].astype(str).str.strip().str.len() > 20]
posts_embed = posts_embed.rename(columns={text_col: 'pagetext'})

print(f'Posts con texto válido para embedding: {len(posts_embed):,}')
print(f'Usuarios cubiertos: {posts_embed["userid"].nunique():,}')

`embed_users` (usada en la Estrategia A) concatena todos los posts de un usuario en un solo texto y genera un embedding — más contexto de estilo. `compute_actor_centroids` (usada en las estrategias C/D/E) embebe cada post por separado y promedia los vectores L2-normalizados — más robusto con corpus grandes o posts atípicos.

La versión completa de estos cálculos sobre todo el dataset (no la muestra que corre este notebook en vivo) se precomputó con `scripts/precompute.py embed --strategy embed_users --file "data/Far Right Forum/IronMarch_2019.11.zip"` y `scripts/precompute.py compare --file "data/Far Right Forum/IronMarch_2019.11.zip" --reference centroids` (este último compara distintos tamaños de muestra). Ver [`scripts/README.md`](../scripts/README.md).

In [ ]:
# Estrategia A: concatenar todos los posts de cada usuario → 1 vector por usuario
# Tiempo estimado: ~12 min (usa caché si ya está calculado)
MAX_CHARS_A = 50_000  # Limitar a 50K chars para evitar timeouts en Ollama

cached_a = load_latest('s5a_embed_users_*.npz')

if cached_a is not None:
    user_ids_a = cached_a['user_ids'].tolist()
    vectors_a  = cached_a['vectors']
    print(f'Parte A cargada desde caché: {len(user_ids_a):,} usuarios, dim={vectors_a.shape[1]}')
else:
    print('Caché no encontrado. Ejecutando Parte A (embed_users)...')
    print(f'Tiempo estimado: ~12 minutos. Dejar corriendo.')

    grouped = (
        posts_embed.groupby('userid')['pagetext']
        .apply(lambda texts: ' '.join(texts.tolist())[:MAX_CHARS_A])
    )
    uids  = grouped.index.tolist()
    texts = grouped.tolist()

    results = []
    batches  = [texts[i:i+32] for i in range(0, len(texts), 32)]  # Procesar de 32 en 32
    for batch in tqdm(batches, desc='Embedding qwen3-embedding'):
        resp = ollama.embed(model='qwen3-embedding', input=batch)
        results.extend(resp.embeddings)

    user_ids_a = uids
    vectors_a  = np.array(results, dtype=np.float32)
    save_timestamped(
        {'user_ids': np.array(user_ids_a, dtype='U64'), 'vectors': vectors_a},
        's5a', 'embed_users',
    )
    print(f'Parte A completada: {len(user_ids_a):,} usuarios')

In [ ]:
# Estrategias C, D, E: top-N posts más largos por usuario
# Tiempo estimado: C ~2.5h, D ~3.5h, E ~4h (usan caché si ya está calculado)

def build_sampled_centroids(max_posts: int, section: str, cache_pattern: str):
    """Construye embeddings tomando los max_posts posts más largos de cada usuario.
    Si ya existe caché, lo carga en lugar de recalcular."""
    cached = load_latest(cache_pattern)
    if cached is not None:
        ids  = cached['user_ids'].tolist()
        vecs = cached['vectors']
        print(f'Cargado (top-{max_posts}): {len(ids):,} usuarios, dim={vecs.shape[1]}')
        return ids, vecs

    # Seleccionar top-N posts más largos por usuario
    sample = (
        posts_embed
        .assign(text_len=posts_embed['pagetext'].str.len())
        .sort_values('text_len', ascending=False)
        .groupby('userid')
        .head(max_posts)
        .drop(columns='text_len')
        .reset_index(drop=True)
    )
    print(f'Computando (top-{max_posts}): {len(sample):,} posts...')
    ids, vecs = compute_actor_centroids(sample, min_posts=5)
    save_timestamped(
        {'user_ids': np.array(ids, dtype='U64'), 'vectors': vecs},
        section, f'centroids_sampled{max_posts}',
    )
    print(f'Completado (top-{max_posts}): {len(ids):,} usuarios')
    return ids, vecs

user_ids_c, vectors_c = build_sampled_centroids(50,  's5c', 's5c_centroids_sampled_*.npz')
user_ids_d, vectors_d = build_sampled_centroids(100, 's5d', 's5d_centroids_sampled100_*.npz')
user_ids_e, vectors_e = build_sampled_centroids(150, 's5e', 's5e_centroids_sampled150_*.npz')

### 2.1 Comparativa Spearman ρ entre estrategias

Para saber si el muestreo (C/D/E) preserva la información del método completo (B),
calculamos la **correlación de Spearman (ρ)** entre las matrices de similitud.

¿Qué significa esto? Para cada par de usuarios, calculamos cuán similares son según
el método B (completo) y según el método C/D/E (muestreado). Si el ranking de similitudes
es prácticamente el mismo, ρ ≈ 1. Si son completamente distintos, ρ ≈ 0.

In [ ]:
# Intentar cargar la estrategia B (completa) desde caché
cached_b = load_latest('s5b_centroids_full_*.npz')
if cached_b is not None:
    user_ids_b = cached_b['user_ids'].tolist()
    vectors_b  = cached_b['vectors']
    print(f'Parte B cargada: {len(user_ids_b):,} usuarios, dim={vectors_b.shape[1]}')
else:
    print('Parte B (centroids_full) no disponible en caché.')
    print('Este cálculo tarda ~9 horas. Los resultados ya calculados están en _archive/03_embeddings_profiling.ipynb.')
    user_ids_b = None; vectors_b = None

def pairwise_sims(user_ids, vectors):
    """Calcula similitudes coseno para todos los pares posibles de usuarios.
    Retorna una Serie de pandas con índice (user_a, user_b) y valor = similitud."""
    sim_matrix = cosine_similarity(vectors)
    np.fill_diagonal(sim_matrix, 0)  # Excluir un usuario consigo mismo
    idx_i, idx_j = np.triu_indices(len(user_ids), k=1)  # Solo triángulo superior
    keys = pd.MultiIndex.from_arrays([
        [str(user_ids[i]) for i in idx_i],
        [str(user_ids[j]) for j in idx_j],
    ])
    return pd.Series(sim_matrix[idx_i, idx_j], index=keys)

available = {}
for label, ids_var, vecs_var in [
    ('A: embed_users',    user_ids_a if 'user_ids_a' in dir() else None, vectors_a if 'vectors_a' in dir() else None),
    ('B: centroids_full', user_ids_b, vectors_b),
    ('C: sampled_top50',  user_ids_c if 'user_ids_c' in dir() else None, vectors_c if 'vectors_c' in dir() else None),
    ('D: sampled_top100', user_ids_d if 'user_ids_d' in dir() else None, vectors_d if 'vectors_d' in dir() else None),
    ('E: sampled_top150', user_ids_e if 'user_ids_e' in dir() else None, vectors_e if 'vectors_e' in dir() else None),
]:
    if ids_var is not None and vecs_var is not None and len(ids_var) > 1:
        print(f'Calculando pares para {label}...')
        available[label] = pairwise_sims(ids_var, vecs_var)

print('\nCorrelaciones de Spearman vs método B (referencia completa):')
ref_label = 'B: centroids_full'
if ref_label in available:
    for lbl, sims in available.items():
        if lbl == ref_label: continue
        common = sims.index.intersection(available[ref_label].index)
        if len(common) == 0: continue
        rho, p = spearmanr(sims[common], available[ref_label][common])
        print(f'  {lbl:<25} ρ={rho:.4f}  (n={len(common):,} pares)')
else:
    print('  Parte B no disponible en caché. Ver resultados en _archive/03_embeddings_profiling.ipynb.')
    print('  Resultados pre-calculados: C→ρ=0.799, D→ρ=0.860, E→ρ=0.896')

## 3. UMAP + HDBSCAN: detectar clusters de usuarios

Tenemos vectores de 4096 dimensiones por usuario — imposible visualizar directamente.

**UMAP** (Uniform Manifold Approximation and Projection) reduce esas 4096 dimensiones
a 2 dimensiones preservando la estructura local: si dos usuarios son similares en 4096D,
seguirán siendo cercanos en 2D.

**HDBSCAN** (Hierarchical Density-Based Spatial Clustering) detecta automáticamente grupos
de usuarios cercanos en el espacio reducido. A diferencia de k-means, no requiere que
especifiquemos cuántos grupos queremos — los detecta según la densidad.

**Parámetro clave:** `min_cluster_size=5` le dice a HDBSCAN que necesita al menos
5 usuarios para considerar algo un cluster real. Usuarios aislados quedan marcados como
ruido (cluster -1, en gris).

In [ ]:
# Elegir el mejor embedding disponible (B > E > D > C > A)
_priority = [
    ('B: centroids_full',  user_ids_b if 'user_ids_b' in dir() else None, vectors_b if 'vectors_b' in dir() else None),
    ('E: sampled_top150',  user_ids_e if 'user_ids_e' in dir() else None, vectors_e if 'vectors_e' in dir() else None),
    ('D: sampled_top100',  user_ids_d if 'user_ids_d' in dir() else None, vectors_d if 'vectors_d' in dir() else None),
    ('C: sampled_top50',   user_ids_c if 'user_ids_c' in dir() else None, vectors_c if 'vectors_c' in dir() else None),
    ('A: embed_users',     user_ids_a if 'user_ids_a' in dir() else None, vectors_a if 'vectors_a' in dir() else None),
]

best_label, best_ids, best_vecs = None, None, None
for label, ids, vecs in _priority:
    if ids is not None and vecs is not None and len(ids) > 1:
        best_label, best_ids, best_vecs = label, ids, vecs
        break

if best_label:
    print(f'Usando embedding: {best_label}')
    print(f'  Usuarios: {len(best_ids):,}  |  Dimensiones: {best_vecs.shape[1]}')

    # UMAP: reducir de 4096D a 2D
    # n_neighbors=15: cada punto considera sus 15 vecinos más cercanos para el layout
    print('Calculando UMAP 2D...')
    reducer   = umap_lib.UMAP(n_components=2, n_neighbors=min(15, len(best_ids) - 1), random_state=42)
    coords_2d = reducer.fit_transform(best_vecs)

    # HDBSCAN: detectar clusters en el espacio de embeddings (no en 2D)
    # min_cluster_size=5: un grupo debe tener al menos 5 usuarios para ser cluster
    # min_samples=3: cuántos vecinos mínimos necesita un punto para no ser ruido
    print('Calculando HDBSCAN...')
    clusterer = hdbscan.HDBSCAN(min_cluster_size=5, min_samples=3)
    clusters  = clusterer.fit_predict(best_vecs)  # Predecir sobre vectores originales 4096D
    n_clusters = len(set(clusters)) - (1 if -1 in clusters else 0)
    print(f'Clusters encontrados: {n_clusters}  |  Ruido (sin cluster): {(clusters == -1).sum()}')
else:
    print('Sin embeddings disponibles. Ejecutar las celdas de embeddings primero.')

In [ ]:
if best_vecs is not None and 'clusters' in dir():
    names = [str(uid_to_name.get(uid, uid)) for uid in best_ids]

    fig = go.Figure(go.Scatter(
        x=coords_2d[:, 0], y=coords_2d[:, 1],
        mode='markers',
        marker=dict(
            size=6,
            color=clusters,
            colorscale='Plasma',
            opacity=0.85,
            showscale=True,
            colorbar=dict(title='Cluster'),
            line=dict(width=0),
        ),
        text=[f'<b>{n}</b><br>cluster={c}' for n, c in zip(names, clusters)],
        hovertemplate='%{text}<extra></extra>',
    ))
    fig.update_layout(
        title=f'UMAP + HDBSCAN — {best_label} ({n_clusters} clusters)<br>'
              '<sup>Cada punto es un usuario. Color = cluster. Gris = sin cluster (ruido). Hover = nombre.</sup>',
        template='plotly_dark', height=620, width=900,
        xaxis=dict(showticklabels=False, showgrid=False, zeroline=False),
        yaxis=dict(showticklabels=False, showgrid=False, zeroline=False),
    )
    fig.show()

    # Mostrar miembros de cada cluster
    cluster_ids = sorted(set(clusters) - {-1})
    print(f'\nMiembros por cluster (excluyendo ruido):')
    for cid in cluster_ids:
        members = [str(uid_to_name.get(uid, uid)) for uid, cl in zip(best_ids, clusters) if cl == cid]
        print(f'  Cluster {cid} ({len(members)} usuarios): {members[:8]}{"..." if len(members) > 8 else ""}')

## 4. BERTopic — apoyo, no protagonista

BERTopic aparece aquí solo como evidencia de apoyo — el protagonista de este caso es la
red de influencia (notebook 02), no el descubrimiento de temas. El hallazgo que sí importa
retener: los topics automáticos **no respetan las fronteras de los subforos declarados**
— la ideología permea todo el foro, no queda contenida en secciones específicas.

In [ ]:
text_col = 'pagetext' if 'pagetext' in posts_clean.columns else 'message'

# Preparar corpus para BERTopic
# Usamos hasta 10.000 posts para que sea computacionalmente manejable
corpus_df = posts_clean[[text_col, 'userid']].copy()
corpus_df = corpus_df.dropna(subset=[text_col])
corpus_df['text_clean'] = (
    corpus_df[text_col].astype(str)
    .str.replace(r'<[^>]+>', ' ', regex=True)  # Quitar HTML
    .str.strip()
)
corpus_df = corpus_df[corpus_df['text_clean'].str.len() > 100]

# Muestra de los posts más largos (más informativos para topic modeling)
if len(corpus_df) > 10000:
    corpus_df = corpus_df.nlargest(10000, 'text_clean')

documents = corpus_df['text_clean'].tolist()
print(f'Documentos para BERTopic: {len(documents):,}')

In [ ]:
# Ajustar BERTopic
# stop_words='english': ignorar palabras comunes en inglés (the, a, is, etc.)
# min_df=5: una palabra debe aparecer en al menos 5 documentos para contar
# ngram_range=(1,2): usar palabras sueltas Y pares de palabras como términos
vectorizer = CountVectorizer(stop_words='english', min_df=5, ngram_range=(1, 2))

topic_model = BERTopic(
    vectorizer_model=vectorizer,
    min_topic_size=15,
    nr_topics='auto',  # Dejar que el modelo decida cuántos topics
    calculate_probabilities=False,
    verbose=True,
)

topics, _ = topic_model.fit_transform(documents)
corpus_df = corpus_df.copy()
corpus_df['topic'] = topics

n_topics = len(set(topics)) - (1 if -1 in topics else 0)
print(f'\nTopics encontrados: {n_topics}')
print(f'Posts sin topic asignado (-1): {topics.count(-1):,}  '
      f'({topics.count(-1)/len(topics)*100:.1f}%)')

# Tabla resumen: topic_id, top words, cantidad de posts
topic_info = topic_model.get_topics()
valid_topics = {t: words for t, words in topic_info.items() if t != -1}
topic_sizes  = {t: topics.count(t) for t in valid_topics}

print('\nTop 15 topics por tamaño:')
print(f'{"ID":>4}  {"Palabras clave":<55}  {"Posts":>6}')
print('-' * 70)
for t in sorted(topic_sizes, key=topic_sizes.get, reverse=True)[:15]:
    kws = ', '.join([w for w, _ in valid_topics[t][:6]])
    print(f'{t:>4}  {kws:<55}  {topic_sizes[t]:>6}')

## 5. NER: extracción de entidades con qwen2.5:14b

**NER** (Named Entity Recognition) identifica entidades nombradas en el texto:
personas, organizaciones, lugares, ideologías, armas, eventos.

Usamos `qwen2.5:14b` via Ollama con un **prompt de zero-shot**: le pedimos al modelo
que identifique entidades sin haberlo entrenado específicamente para este dataset.
El resultado es un JSON con las entidades encontradas y su tipo.

**Estrategia de muestreo:** top 50 usuarios por volumen de posts, hasta 50 posts por usuario.
Esto nos da ~2.500 posts a analizar — representativo sin ser prohibitivo en tiempo.

**Caché:** los resultados de NER se guardan en `ner_results.parquet`.
Si el archivo existe, se carga directamente — el NER puede tardar 2-3 horas.

In [ ]:
SYSTEM_PROMPT = """You are an expert forensic analyst. Extract named entities from forum posts.
Return ONLY valid JSON object: {"entities": [{"text": "...", "type": "PERSON|ORG|LOCATION|IDEOLOGY|WEAPON|EVENT", "confidence": 0.0-1.0}]}"""

def extract_entities(text: str, model: str = 'qwen2.5:14b') -> list:
    """Extrae entidades nombradas de un post usando qwen2.5:14b vía Ollama.
    Retorna una lista de dicts con 'text', 'type' y 'confidence'."""
    try:
        response = ollama.generate(
            model=model,
            system=SYSTEM_PROMPT,
            prompt=str(text)[:1500],  # Limitar a 1500 chars por post
            format='json',
            options={'temperature': 0},  # 0 = respuesta determinista
        )
        result   = json.loads(response['response'])
        entities = result.get('entities', []) if isinstance(result, dict) else []
        return entities if isinstance(entities, list) else []
    except Exception:
        return []

# Muestra: top 50 usuarios por cantidad de posts, máximo 50 posts cada uno
posts_text = posts_clean.dropna(subset=[text_col]).copy()
posts_text = posts_text[posts_text[text_col].astype(str).str.len() > 100]
top_user_ids = posts_text.groupby('userid').size().nlargest(50).index.tolist()

sample_top = (
    posts_text[posts_text['userid'].isin(top_user_ids)]
    .groupby('userid')
    .apply(lambda g: g.nlargest(50, text_col) if len(g) > 50 else g)
    .reset_index(drop=True)
)
print(f'Muestra para NER:')
print(f'  Usuarios:     {sample_top["userid"].nunique()}')
print(f'  Posts totales: {len(sample_top):,}')

La caché de `ner_results.parquet` la generó `scripts/precompute.py ner --file "data/Far Right Forum/IronMarch_2019.11.zip" --sample-size 500`, que extrae entidades (personas, organizaciones, ideología, etc.) usando un LLM local vía Ollama sobre todo el corpus.

Es la plantilla a reutilizar si quieres extraer entidades de dominio (IPs, handles, herramientas) de tu propio dataset con un LLM local — ver [`scripts/README.md`](../scripts/README.md).

In [ ]:
NER_CACHE = IM_RESULTS / 'ner_results.parquet'

if NER_CACHE.exists():
    ner_all = pd.read_parquet(NER_CACHE)
    print(f'NER cargado desde caché: {len(ner_all):,} entidades')
    print(f'Tipos de entidad: {ner_all["type"].value_counts().to_dict()}')
else:
    print('Ejecutando NER...')
    print('Tiempo estimado: 2-3 horas. Los resultados se guardan con checkpoint cada 10 usuarios.')
    all_records = []
    user_list   = sample_top['userid'].unique().tolist()

    for i, uid in enumerate(tqdm(user_list, desc='NER por usuario')):
        user_posts = sample_top[sample_top['userid'] == uid]
        for _, row in user_posts.iterrows():
            entities = extract_entities(str(row[text_col]))
            for ent in entities:
                txt = str(ent.get('text', '')).strip()
                if len(txt) < 2: continue
                all_records.append({
                    'userid':     uid,
                    'entity':     txt,
                    'type':       ent.get('type', 'UNKNOWN').upper(),
                    'confidence': float(ent.get('confidence', 0.5)),
                })
        # Checkpoint cada 10 usuarios
        if (i + 1) % 10 == 0 and all_records:
            pd.DataFrame(all_records).to_parquet(NER_CACHE, index=False)
            print(f'  Checkpoint: {i+1}/{len(user_list)} usuarios procesados')

    if all_records:
        ner_all = pd.DataFrame(all_records)
        ner_all = ner_all[ner_all['entity'].str.len() > 1]
        ner_all.to_parquet(NER_CACHE, index=False)
        print(f'NER completado: {len(ner_all):,} entidades')
    else:
        print('Sin resultados. Verificar que qwen2.5:14b esté disponible: ollama pull qwen2.5:14b')
        ner_all = pd.DataFrame(columns=['userid', 'entity', 'type', 'confidence'])

In [ ]:
# Visualización: top entidades por tipo
if len(ner_all) > 0:
    type_counts = ner_all['type'].value_counts()
    fig = go.Figure(go.Bar(
        x=type_counts.index.tolist(),
        y=type_counts.values.tolist(),
        marker_color=['#E94E4E', '#4E9EE9', '#E9A84E', '#7AE94E', '#C44EE9', '#FFD700'],
    ))
    fig.update_layout(
        title='Distribución de tipos de entidad — NER IronMarch',
        xaxis_title='Tipo', yaxis_title='Menciones',
        template='plotly_dark', height=380,
    )
    fig.show()

    entity_types = [t for t in ['PERSON', 'IDEOLOGY', 'ORG', 'ORGANIZATION', 'LOCATION', 'EVENT', 'WEAPON']
                    if t in ner_all['type'].unique()]

    if entity_types:
        n_cols = min(len(entity_types), 3)
        n_rows = (len(entity_types) + n_cols - 1) // n_cols
        fig2   = make_subplots(rows=n_rows, cols=n_cols, subplot_titles=entity_types,
                               horizontal_spacing=0.08, vertical_spacing=0.12)
        colors = ['#E94E4E', '#4E9EE9', '#E9A84E', '#7AE94E', '#C44EE9', '#FFD700', '#FF6B35']
        for i, etype in enumerate(entity_types):
            row, col = i // n_cols + 1, i % n_cols + 1
            top = ner_all[ner_all['type'] == etype]['entity'].value_counts().head(10)
            fig2.add_trace(go.Bar(
                x=top.values[::-1], y=top.index[::-1].tolist(),
                orientation='h', marker_color=colors[i % len(colors)], showlegend=False,
            ), row=row, col=col)
        fig2.update_layout(
            title='Top 10 entidades por tipo — IronMarch NER',
            template='plotly_dark',
            height=380 * n_rows, width=min(380 * n_cols, 1100),
            margin=dict(l=150, r=20, t=60, b=40),
        )
        fig2.show()

## 6. Perfilado de actores con LLM

Esta es una sección nueva que no existía en los notebooks originales.

Para cada usuario relevante (top brokers de la red + usuarios en clusters específicos),
sintetizamos un perfil usando `qwen2.5:14b`. El modelo recibe:
1. El corpus completo del usuario (sus posts más representativos)
2. Sus entidades NER más frecuentes
3. Su topic principal (BERTopic)

Y retorna una síntesis estructurada: **rol probable, ideología, nivel de radicalización,
patrones de comportamiento observados**.

**Importante:** el perfil del LLM es una hipótesis, no un diagnóstico.
Debe tratarse como un punto de partida para análisis manual, no como conclusión definitiva.

In [ ]:
PROFILING_PROMPT = """You are a forensic analyst studying extremist forums. 
Analyze the following information about a forum user and provide a structured profile.
Be analytical and objective. Base conclusions only on evidence provided.

User data:
- Username: {username}
- Most mentioned entities: {entities}
- Main topic cluster: {topic}
- Sample posts: {posts_sample}

Return JSON:
{"role": "ideological_leader|recruiter|moderator|active_member|newcomer|lurker",
 "ideology_indicators": ["list", "of", "observed", "indicators"],
 "behavioral_patterns": ["patterns", "observed"],
 "radicalization_level": "low|medium|high|extreme",
 "confidence": 0.0-1.0,
 "summary": "one paragraph synthesis"
}"""

def profile_user(username: str, entities_str: str, topic_str: str, posts_sample: str) -> dict:
    """Genera un perfil de usuario usando qwen2.5:14b.
    Retorna un diccionario con el perfil o un dict vacío si hay error."""
    prompt = PROFILING_PROMPT.format(
        username=username,
        entities=entities_str,
        topic=topic_str,
        posts_sample=posts_sample[:2000],  # Limitar el contexto
    )
    try:
        response = ollama.generate(
            model='qwen2.5:14b',
            prompt=prompt,
            format='json',
            options={'temperature': 0.1},
        )
        return json.loads(response['response'])
    except Exception as e:
        return {'error': str(e)}

print('Función de perfilado lista.')
print('Verificar modelo disponible:')
import subprocess
subprocess.run(['ollama', 'list'], check=False)

In [ ]:
PROFILES_CACHE = IM_RESULTS / 'actor_profiles.parquet'

if PROFILES_CACHE.exists():
    actor_profiles = pd.read_parquet(PROFILES_CACHE)
    print(f'Perfiles cargados desde caché: {len(actor_profiles)} actores')
else:
    # Seleccionar usuarios a perfilar: top 20 por volumen de posts con datos NER
    users_to_profile = (
        posts_clean.groupby('userid').size()
        .nlargest(20)
        .index.tolist()
    )

    profile_records = []
    corpus_by_user_path = IM_RESULTS / 'corpus_users_clean.parquet'
    if corpus_by_user_path.exists():
        corpus_by_user = pd.read_parquet(corpus_by_user_path)
    else:
        corpus_by_user = None

    for uid in tqdm(users_to_profile, desc='Perfilando actores'):
        uname = str(uid_to_name.get(uid, uid))

        # Entidades más frecuentes de este usuario
        if len(ner_all) > 0 and 'userid' in ner_all.columns:
            user_ents = ner_all[ner_all['userid'] == uid]['entity'].value_counts().head(10)
            entities_str = ', '.join(user_ents.index.tolist())
        else:
            entities_str = 'no disponible'

        # Topic principal del usuario
        if 'topic' in corpus_df.columns and 'userid' in corpus_df.columns:
            user_topics = corpus_df[corpus_df['userid'] == uid]['topic'].value_counts()
            top_topic_id = user_topics.index[0] if len(user_topics) > 0 else -1
            if top_topic_id != -1 and top_topic_id in valid_topics:
                topic_str = ', '.join([w for w, _ in valid_topics[top_topic_id][:5]])
            else:
                topic_str = 'sin topic asignado'
        else:
            topic_str = 'no disponible'

        # Muestra de posts
        if corpus_by_user is not None:
            row = corpus_by_user[corpus_by_user['userid'] == uid]
            posts_sample = row['corpus'].iloc[0][:2000] if len(row) > 0 else ''
        else:
            user_posts = posts_clean[posts_clean['userid'] == uid][text_col].dropna()
            posts_sample = ' '.join(user_posts.head(10).tolist())[:2000]

        # Generar perfil
        profile = profile_user(uname, entities_str, topic_str, posts_sample)
        profile['userid']   = uid
        profile['username'] = uname
        profile_records.append(profile)

    actor_profiles = pd.DataFrame(profile_records)
    actor_profiles.to_parquet(PROFILES_CACHE, index=False)
    print(f'Perfiles generados: {len(actor_profiles)}')

# Mostrar perfiles
display_cols = [c for c in ['username', 'role', 'radicalization_level', 'confidence', 'summary']
                if c in actor_profiles.columns]
if display_cols:
    print('\nPerfiles de actores:')
    for _, row in actor_profiles[display_cols].iterrows():
        print(f'\n  [{row.get("username", "")}]')
        for col in display_cols[1:]:
            print(f'    {col}: {row.get(col, "")}')

## 7. Burrows' Delta: estilometría

**Burrows' Delta** es el estándar en **atribución de autoría** — la técnica que usan los
académicos para determinar si dos textos fueron escritos por la misma persona.

**¿Cómo funciona?**
En lugar de analizar el tema o el vocabulario especializado, mide las **palabras función**:
preposiciones, artículos, conjunciones, pronombres. Estas palabras se usan de forma
inconsciente e idiosincrática — cada persona tiene un patrón único que no cambia aunque
escriba sobre temas distintos.

El Delta entre dos usuarios es la diferencia media en z-score de esas palabras función.
**Delta bajo = estilo similar = posible misma persona (sockpuppet)**.

**Limitación importante:** un corpus homogéneo (todos los usuarios hablan sobre los mismos
temas extremistas) reduce el poder discriminante. Los candidatos son *leads*, no pruebas.

Existe una implementación de referencia de estilometría en `src/stylometry.py` (`extract_features`, `compare_users`), pero **no** es la que se usa aquí: el Burrows' Delta de este caso se calculó con un método distinto, específico de este notebook, y no tiene código fuente reutilizable en el repo más allá de las celdas siguientes.

In [ ]:
# Preparar corpus para Burrows' Delta
# Mínimo 500 palabras por usuario — menos texto = Delta estadísticamente poco fiable
MIN_WORDS = 500

def clean_for_delta(text: str) -> str:
    """Limpiar texto para Delta: quitar HTML, URLs, y dejar solo letras en minúsculas."""
    text = re.sub(r'<[^>]+>', ' ', str(text))
    text = re.sub(r'https?://\S+', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    return text.lower()

records = []
for uid, group in posts_clean.groupby('userid'):
    uname = uid_to_name.get(uid, uid)
    if pd.isna(uname): continue
    combined = ' '.join(group[text_col].dropna().astype(str).tolist())
    words    = clean_for_delta(combined).split()
    if len(words) >= MIN_WORDS:
        records.append({'userid': uid, 'username': str(uname), 'words': words, 'n_words': len(words)})

delta_corpus = pd.DataFrame(records).sort_values('n_words', ascending=False).reset_index(drop=True)
print(f'Usuarios con ≥{MIN_WORDS} palabras para Delta: {len(delta_corpus):,}')
print(f'Mediana de palabras por usuario: {delta_corpus["n_words"].median():,.0f}')
print(f'\nTop 10 usuarios con más texto:')
print(delta_corpus[['username', 'n_words']].head(10).to_string(index=False))

In [ ]:
# Calcular Burrows' Delta — top 200 palabras más frecuentes, top 150 usuarios
N_FEATURES = 200  # Palabras función más comunes del corpus
TOP_USERS  = 150  # Analizar los 150 usuarios con más texto

# Obtener vocabulario de las N palabras más frecuentes
all_words = [w for row in delta_corpus['words'] for w in row]
vocab     = [w for w, _ in Counter(all_words).most_common(N_FEATURES)]
vocab_set = set(vocab)
print(f'Top {N_FEATURES} palabras función: {" | ".join(vocab[:20])} ...')

top_users_delta = delta_corpus.head(TOP_USERS).copy()

# Construir matriz de frecuencias relativas
# freq_matrix[i][j] = frecuencia de la palabra j en el corpus del usuario i
freq_matrix = np.zeros((len(top_users_delta), N_FEATURES))
for i, (_, row) in enumerate(top_users_delta.iterrows()):
    counts = Counter(w for w in row['words'] if w in vocab_set)
    total  = row['n_words']
    for j, w in enumerate(vocab):
        freq_matrix[i, j] = counts.get(w, 0) / total

# Normalización por z-score por columna (por palabra)
# Cada palabra queda en la misma escala — palabras raras no dominan el resultado
mu_v    = freq_matrix.mean(axis=0)
sigma_v = freq_matrix.std(axis=0)
sigma_v[sigma_v == 0] = 1  # Evitar división por cero
z_matrix = (freq_matrix - mu_v) / sigma_v

# Delta = promedio de |z_i - z_j| para todos los features
n = len(top_users_delta)
delta_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(i + 1, n):
        d = np.mean(np.abs(z_matrix[i] - z_matrix[j]))
        delta_matrix[i, j] = d
        delta_matrix[j, i] = d

delta_df_matrix = pd.DataFrame(
    delta_matrix,
    index=top_users_delta['username'].tolist(),
    columns=top_users_delta['username'].tolist(),
)

vals = delta_matrix[delta_matrix > 0]
print(f'\nMatriz Delta: {n}×{n}')
print(f'Delta medio: {vals.mean():.4f}  |  mínimo: {vals.min():.4f}  |  máximo: {vals.max():.4f}')

In [ ]:
# Heatmap interactivo: top 40 usuarios por Burrows' Delta
# Verde oscuro = Delta bajo = estilos similares = candidatos a misma persona
TOP_HEAT = 40
sub    = delta_df_matrix.iloc[:TOP_HEAT, :TOP_HEAT]
labels = sub.index.tolist()

fig = go.Figure(go.Heatmap(
    z=sub.values,
    x=labels, y=labels,
    colorscale='RdYlGn',
    reversescale=True,  # Verde = similar (bajo Delta)
    hovertemplate='%{y} vs %{x}<br>Delta=%{z:.4f}<extra></extra>',
    colorbar=dict(title='Burrows Delta<br>(↓ = más similar)'),
))
fig.update_layout(
    title="Burrows' Delta — top 40 usuarios IronMarch<br>"
          "<sup>Verde oscuro = estilo similar → candidato a misma persona (sockpuppet)</sup>",
    template='plotly_dark', height=720, width=760,
    xaxis=dict(tickangle=-45, tickfont=dict(size=8)),
    yaxis=dict(tickfont=dict(size=8)),
    margin=dict(l=140, r=20, t=80, b=140),
)
fig.show()

In [ ]:
# Candidatos a sockpuppet: pares con menor Delta
users_list = delta_df_matrix.index.tolist()
n = len(users_list)
pairs = []
name_to_nwords = dict(zip(delta_corpus['username'], delta_corpus['n_words']))

for i in range(n):
    for j in range(i + 1, n):
        pairs.append({
            'user_a':  users_list[i],
            'user_b':  users_list[j],
            'delta':   round(delta_df_matrix.iloc[i, j], 4),
            'words_a': name_to_nwords.get(users_list[i], 0),
            'words_b': name_to_nwords.get(users_list[j], 0),
        })

pairs_df = pd.DataFrame(pairs).sort_values('delta')
q01 = pairs_df['delta'].quantile(0.01)
q10 = pairs_df['delta'].quantile(0.10)

print('Top 20 pares con menor Delta (candidatos a sockpuppet):')
print(pairs_df.head(20)[['user_a', 'user_b', 'delta']].to_string(index=False))
print(f'\nPercentil  1%: Delta < {q01:.4f}')
print(f'Percentil 10%: Delta < {q10:.4f}')

suspicious = pairs_df[pairs_df['delta'] <= q01]
print(f'\nTop candidatos (percentil 1%, {len(suspicious)} pares):')
print(suspicious[['user_a', 'user_b', 'delta']].to_string(index=False))

In [ ]:
# Guardar todas las señales semánticas para el notebook de síntesis
# Tabla de señales: una fila por usuario con todas las métricas computadas

semantic_signals = users_clean[['userid', 'username_raw']].copy()
semantic_signals = semantic_signals.rename(columns={'username_raw': 'username'})

# Agregar cluster HDBSCAN
if 'clusters' in dir() and best_ids is not None:
    cluster_map = dict(zip(best_ids, clusters))
    semantic_signals['hdbscan_cluster'] = semantic_signals['userid'].map(cluster_map)

# Agregar top topic BERTopic por usuario
if 'topic' in corpus_df.columns:
    top_topic_per_user = (
        corpus_df[corpus_df['topic'] != -1]
        .groupby('userid')['topic']
        .agg(lambda x: x.value_counts().idxmax())
        .reset_index()
        .rename(columns={'topic': 'top_topic'})
    )
    semantic_signals = semantic_signals.merge(top_topic_per_user, on='userid', how='left')

# Agregar min_delta (cuán similar estilísticamente es al usuario más parecido)
if 'delta_df_matrix' in dir() and len(delta_df_matrix) > 0:
    username_to_uid = dict(zip(users_clean['username_raw'], users_clean['userid']))
    min_delta_map = {}
    for uname in delta_df_matrix.index:
        vals_row = delta_df_matrix.loc[uname]
        vals_row = vals_row[vals_row > 0]
        min_delta_map[uname] = vals_row.min() if len(vals_row) > 0 else np.nan
    semantic_signals['min_delta'] = semantic_signals['username'].map(min_delta_map)

out_path = IM_RESULTS / 'semantic_signals.parquet'
semantic_signals.to_parquet(out_path, index=False)
print(f'Señales semánticas guardadas: {out_path}')
print(f'  {len(semantic_signals):,} usuarios')
print(f'  Columnas: {list(semantic_signals.columns)}')
print(f'\nSiguiente paso: 04_sintesis_informe.ipynb')